In [ ]:
# ==============================================================================
# NEXUS OPTIML 2026: END-TO-END PRODUCT RETURN PREDICTION PIPELINE
# ARCHITECTURE: MULTI-TABLE IMPUTATION + BRACKETING + ADAPTIVE OOF TE + 
#               ROLLING PURGED CV + ENSEMBLE (LGBM + CATBOOST + RF + SGD)
# ==============================================================================

import gc
import os
import time
import warnings
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold

# Gradient Boosted Decision Tree Frameworks
import lightgbm as lgb

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 120)

SEED = 42
DATA_DIR = '.'  # Path to competition CSV files
N_JOBS = -1

print(f"LightGBM Version  : {lgb.__version__}")
print(f"CatBoost Ready    : {CATBOOST_AVAILABLE}")
print(f"XGBoost Ready     : {XGBOOST_AVAILABLE}")


# ==============================================================================
# STEP 1: CLEANING & LOGICAL IMPUTATION ACROSS SUPPLEMENTARY TABLES
# ==============================================================================
print("\n--- [Step 1/7] Cleaning & Logically Imputing Supplementary Tables ---")

def clean_region_str(series: pd.Series) -> pd.Series:
    """Normalizes whitespace, underscores, and regional casing."""
    return (
        series.astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r'[\s_-]+', '-', regex=True)
    )

# 1. CUSTOMERS: Deduplicate snapshots and clean regions
df_cust = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
df_cust['region'] = clean_region_str(df_cust['region'])
df_cust['record_updated_at'] = pd.to_datetime(df_cust['record_updated_at'], errors='coerce')
df_cust = df_cust.sort_values('record_updated_at').drop_duplicates(subset=['customer_id'], keep='last')

df_cust['age_band_is_missing'] = df_cust['age_band'].isnull().astype(np.int8)
df_cust['age_band'] = df_cust['age_band'].fillna('unknown')
df_cust['preferred_contact'] = df_cust['preferred_contact'].fillna('none')
df_cust['newsletter_opt_in'] = df_cust['newsletter_opt_in'].fillna(0).astype(np.int8)

# 2. PRODUCTS: Canonicalize categories (threshold >= 300 preserves all 12 true retail categories)
df_prod = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
clean_cat = df_prod['category'].astype(str).str.lower().str.replace(r'[^a-z]', '', regex=True)
cat_counts = clean_cat.value_counts()
canonical_cats = cat_counts[cat_counts >= 300].index.tolist()
cat_map = {
    c: (c if c in canonical_cats else (difflib.get_close_matches(c, canonical_cats, n=1, cutoff=0.5) or [c])[0])
    for c in cat_counts.index
}
df_prod['category_clean'] = clean_cat.map(cat_map)

# Volumetric dimensions (cm^3)
df_prod['vol_cm3'] = df_prod['length_cm'] * df_prod['width_cm'] * df_prod['height_cm']

# Physical density (kg / cm^3) per category to logically recover missing weight
known_weights = df_prod.dropna(subset=['weight_kg']).copy()
known_weights['density'] = known_weights['weight_kg'] / (known_weights['vol_cm3'] + 1e-5)
cat_density = known_weights.groupby('category_clean')['density'].median()
overall_density = known_weights['density'].median()

missing_weight = df_prod['weight_kg'].isnull()
df_prod.loc[missing_weight, 'weight_kg'] = (
    df_prod.loc[missing_weight, 'vol_cm3'] *
    df_prod.loc[missing_weight, 'category_clean'].map(cat_density).fillna(overall_density)
)
df_prod['weight_is_imputed'] = missing_weight.astype(np.int8)

# Category-relative price and weight benchmarks
cat_benchmarks = df_prod.groupby('category_clean').agg(
    cat_median_price=('list_price', 'median'),
    cat_median_weight=('weight_kg', 'median')
).reset_index()
df_prod = df_prod.merge(cat_benchmarks, on='category_clean', how='left')
df_prod['price_to_cat_median'] = df_prod['list_price'] / (df_prod['cat_median_price'] + 1e-4)
df_prod['weight_to_cat_median'] = df_prod['weight_kg'] / (df_prod['cat_median_weight'] + 1e-4)

# Explicitly drop raw uncleaned category column to prevent type conversion errors downstream
df_prod.drop(columns=['category'], inplace=True)

# 3. STORES: Retail format-driven floor space imputation
df_stores = pd.read_csv(os.path.join(DATA_DIR, 'stores.csv'))
df_stores['region'] = clean_region_str(df_stores['region'])
df_stores['is_physical_store'] = df_stores['channel'].isin(['store', 'catalog']).astype(np.int8)

format_size_median = df_stores[df_stores['is_physical_store'] == 1].groupby('format')['size_sqm'].median()
for fmt, med_size in format_size_median.items():
    mask = (df_stores['format'] == fmt) & (df_stores['size_sqm'].isnull())
    df_stores.loc[mask, 'size_sqm'] = med_size
df_stores['size_sqm'] = df_stores['size_sqm'].fillna(0.0)

# 4. CALENDAR: Ingestion and date alignment
df_cal = pd.read_csv(os.path.join(DATA_DIR, 'calendar.csv'))
df_cal['week_start'] = pd.to_datetime(df_cal['week_start'])

print(f"Products missing weight after density recovery: {df_prod['weight_kg'].isnull().sum()}")
print(f"Stores missing size_sqm after format recovery   : {df_stores['size_sqm'].isnull().sum()}")
print(f"Customers missing age_band after tagging        : {df_cust['age_band'].isnull().sum()}")


# ==============================================================================
# STEP 2: ADVANCED FEATURE EXTRACTION & BRACKETING DETECTORS
# ==============================================================================
print("\n--- [Step 2/7] Constructing Advanced Behavioral & Bracketing Features ---")

def extract_advanced_features(df: pd.DataFrame) -> pd.DataFrame:
    df['order_ts'] = pd.to_datetime(df['order_ts'])
    # Fast Monday week alignment (vectorized)
    df['week_start'] = df['order_ts'].dt.floor('D') - pd.to_timedelta(df['order_ts'].dt.dayofweek, unit='D')

    # Relational joins
    df = df.merge(df_cust, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(df_prod, on='product_id', how='left', suffixes=('', '_prod'))
    df = df.merge(df_stores, on='store_id', how='left', suffixes=('', '_store'))
    df = df.merge(df_cal, on='week_start', how='left')

    # Fix price scaling artifact (cents vs. dollars in test set)
    ratio = df['unit_price'] / (df['list_price'] + 1e-4)
    df['is_price_in_cents'] = ((ratio >= 60.0) & (ratio <= 140.0)).astype(np.int8)
    df['unit_price_clean'] = np.where(df['is_price_in_cents'] == 1, df['unit_price'] / 100.0, df['unit_price']).astype(np.float32)

    # Impute missing order attributes logically
    channel_is_phys = df['channel'].isin(['store', 'catalog'])
    df['tz_hint'] = np.where(df['tz_hint'].isnull(), np.where(channel_is_phys, 'local', 'UTC'), df['tz_hint'])
    df['ship_mode'] = np.where(df['ship_mode'].isnull(), np.where(df['channel'] == 'store', 'in_store', 'standard'), df['ship_mode'])
    df['payment_method'] = df['payment_method'].fillna('card')
    df['discount'] = df['discount'].fillna(0.0)
    df['session_clicks'] = df['session_clicks'].fillna(df['session_clicks'].median())

    # Deterministic logistics freight calculation
    freight_formula = (df['weight_kg'] * 2.5 + 5.0).clip(1.0, 50.0)
    df['freight_cost'] = df['freight_cost'].fillna(freight_formula)

    # Stationary cyclical temporal representations
    df['order_month'] = df['order_ts'].dt.month.astype(np.int8)
    df['order_dayofweek'] = df['order_ts'].dt.dayofweek.astype(np.int8)
    df['order_hour'] = df['order_ts'].dt.hour.astype(np.int8)
    df['is_weekend'] = df['order_dayofweek'].isin([5, 6]).astype(np.int8)
    df['is_month_end'] = (df['order_ts'].dt.day >= 26).astype(np.int8)
    df['date_only_flag'] = (df['order_ts'].dt.minute == 1).astype(np.int8)

    df['sin_month'] = np.sin(2 * np.pi * df['order_month'] / 12.0).astype(np.float32)
    df['cos_month'] = np.cos(2 * np.pi * df['order_month'] / 12.0).astype(np.float32)
    df['sin_hour'] = np.sin(2 * np.pi * df['order_hour'] / 24.0).astype(np.float32)
    df['cos_hour'] = np.cos(2 * np.pi * df['order_hour'] / 24.0).astype(np.float32)

    # Pricing & discount metrics
    df['line_raw_val'] = (df['unit_price_clean'] * df['quantity']).astype(np.float32)
    df['calc_line_total'] = (df['line_raw_val'] * (1.0 - df['discount'])).astype(np.float32)
    df['price_ratio_list'] = (df['unit_price_clean'] / (df['list_price'] + 1e-4)).astype(np.float32)
    df['discount_diff'] = (df['list_price'] - df['unit_price_clean']).clip(lower=0).astype(np.float32)
    df['is_deep_discount'] = (df['discount'] >= 0.35).astype(np.int8)

    # Return window exposure
    df['window_price_exposure'] = (df['return_window_days'] * np.log1p(df['unit_price_clean'].clip(lower=0))).astype(np.float32)
    df['window_x_discount'] = (df['return_window_days'] * df['discount']).astype(np.float32)

    # Logistics friction
    df['freight_share'] = (df['freight_cost'] / (df['order_value'] + 1e-4)).astype(np.float32)
    df['freight_per_unit'] = (df['freight_cost'] / (df['quantity'] + 1e-4)).astype(np.float32)
    df['freight_per_kg'] = (df['freight_cost'] / (df['weight_kg'] + 1e-4)).astype(np.float32)
    df['cross_region_shipping'] = (df['region'] != df['region_store']).astype(np.int8)

    # Session engagement
    df['has_coupon'] = df['coupon_code'].notnull().astype(np.int8)
    df['clicks_per_item'] = (df['session_clicks'] / (df['items_in_order'] + 1e-4)).astype(np.float32)
    df['line_position_ratio'] = (df['line_no'] / (df['items_in_order'] + 1e-4)).astype(np.float32)

    # -------------------------------------------------------------------------
    # BRACKETING DETECTION (WARDROBING / MULTI-ITEM COLOR & SIZE SPREAD)
    # -------------------------------------------------------------------------
    order_grp = df.groupby('order_id')
    df['order_line_count'] = order_grp['order_line_id'].transform('count').astype(np.int16)
    df['order_total_qty'] = order_grp['quantity'].transform('sum').astype(np.int32)
    df['order_max_price'] = order_grp['unit_price_clean'].transform('max').astype(np.float32)
    df['order_min_price'] = order_grp['unit_price_clean'].transform('min').astype(np.float32)
    df['order_price_spread'] = (df['order_max_price'] - df['order_min_price']).astype(np.float32)
    df['order_distinct_prods'] = order_grp['product_id'].transform('nunique').astype(np.int16)
    
    # Specific bracketing flags within same order and category
    order_cat_grp = df.groupby(['order_id', 'category_clean'])
    df['items_in_same_cat_in_order'] = order_cat_grp['order_line_id'].transform('count').astype(np.int16)
    df['is_bracketing_category'] = (df['items_in_same_cat_in_order'] > 1).astype(np.int8)

    order_supp_grp = df.groupby(['order_id', 'supplier_code'])
    df['items_from_same_supplier_in_order'] = order_supp_grp['order_line_id'].transform('count').astype(np.int16)

    df['line_qty_share'] = (df['quantity'] / (df['order_total_qty'] + 1e-4)).astype(np.float32)
    df['price_rel_to_max'] = (df['unit_price_clean'] / (df['order_max_price'] + 1e-4)).astype(np.float32)
    df['val_share_order'] = (df['calc_line_total'] / (df['order_value'] + 1e-4)).astype(np.float32)

    # Calendar and macroeconomic context
    df['holiday_flag'] = df['holiday_flag'].fillna(0).astype(np.int8)
    df['temperature_f'] = df['temperature_f'].fillna(60.0).astype(np.float32)
    df['price_to_cpi'] = (df['unit_price_clean'] / (df['cpi'] + 1e-4)).astype(np.float32)
    df['cost_pressure'] = (df['fuel_price'] * (df['fuel_surcharge'].fillna(0) + 1.0)).astype(np.float32)

    # Interaction keys for target encoding
    df['promo_x_channel'] = df['promo_channel_code'].astype(str) + '_' + df['channel'].astype(str)
    df['promo_x_cat'] = df['promo_channel_code'].astype(str) + '_' + df['category_clean'].astype(str)

    # Drop non-stationary datetimes, duplicate columns, and drifting counters
    drop_cols = [
        'signup_date', 'launch_date', 'opened_date', 'week_start',
        'record_updated_at', 'coupon_code', 'warehouse_zone', 'picker_shift'
    ]
    if 'category' in df.columns:
        drop_cols.append('category')
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    return df


# ==============================================================================
# STEP 3: DATA MERGING, UNSUPERVISED STATS & SAFE ENCODING
# ==============================================================================
print("\n--- [Step 3/7] Loading Data & Generating Label-Free Entity Statistics ---")

df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

df_train['is_train'] = 1
df_test['is_train'] = 0
df_test['returned'] = np.nan

df_combined = pd.concat([df_train, df_test], ignore_index=True)
del df_train, df_test
gc.collect()

df_combined = extract_advanced_features(df_combined)

# Unsupervised entity statistics (label-free across train and test)
for col in ['store_id', 'product_id', 'customer_id', 'supplier_code']:
    df_combined[f'{col}_freq'] = df_combined.groupby(col)[col].transform('count').astype(np.int32)

df_combined['cust_orders'] = df_combined.groupby('customer_id')['order_id'].transform('nunique').astype(np.int32)
df_combined['cust_mean_qty'] = df_combined.groupby('customer_id')['quantity'].transform('mean').astype(np.float32)
df_combined['prod_mean_qty'] = df_combined.groupby('product_id')['quantity'].transform('mean').astype(np.float32)

# Excluded non-feature metadata columns
ignore_cols = [
    'order_line_id', 'order_id', 'customer_id', 'product_id', 'store_id',
    'returned', 'is_train', 'order_ts', 'promo_x_channel', 'promo_x_cat', 'supplier_code'
]

# Factorize all non-numeric columns into int32 indices
non_num_cols = [c for c in df_combined.columns if c not in ignore_cols and not pd.api.types.is_numeric_dtype(df_combined[c])]
print(f"Factorizing {len(non_num_cols)} categorical columns to int32...")
for col in non_num_cols:
    df_combined[col] = df_combined[col].astype(str)
    cats, uniques = pd.factorize(df_combined[col])
    df_combined[col] = cats.astype(np.int32)

# Separate train and test splits
df_train = df_combined[df_combined['is_train'] == 1].reset_index(drop=True)
df_test = df_combined[df_combined['is_train'] == 0].reset_index(drop=True)
del df_combined
gc.collect()

base_features = [c for c in df_train.columns if c not in ignore_cols and pd.api.types.is_numeric_dtype(df_train[c])]
print(f"Base Features Extracted: {len(base_features)}")


# ==============================================================================
# STEP 4: EDA VISUALIZATIONS & CORRELATION HEATMAP
# ==============================================================================
print("\n--- [Step 4/7] Generating Exploratory Visualizations & Heatmaps ---")

eda_sample = df_train.sample(n=min(100000, len(df_train)), random_state=SEED)
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. Target Distribution
sns.countplot(x='returned', data=eda_sample, ax=axes[0, 0], palette='Blues')
axes[0, 0].set_title('Target Distribution (0 = Kept, 1 = Returned)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Returned')
axes[0, 0].set_ylabel('Transaction Count')

# 2. Return Rate by Return Window Days
window_returns = df_train.groupby('return_window_days')['returned'].mean().reset_index()
sns.barplot(x='return_window_days', y='returned', data=window_returns, ax=axes[0, 1], palette='crest')
axes[0, 1].set_title('Empirical Return Rate by Return Window (Days)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Return Window (Days)')
axes[0, 1].set_ylabel('Mean Return Probability')

# 3. Cleaned Unit Price vs Return Status (Log Scale)
eda_sample['log_price'] = np.log1p(eda_sample['unit_price_clean'])
sns.boxplot(x='returned', y='log_price', data=eda_sample, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Log Cleaned Unit Price by Return Outcome', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Returned')
axes[1, 0].set_ylabel('Log(1 + Clean Price)')

# 4. Correlation Heatmap
corr_cols = [
    'returned', 'unit_price_clean', 'quantity', 'discount', 'freight_share',
    'window_price_exposure', 'is_bracketing_category', 'line_qty_share', 'order_line_count'
]
corr_matrix = eda_sample[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.3, vmax=0.3, ax=axes[1, 1])
axes[1, 1].set_title('Feature Correlation Heatmap with Target', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


# ==============================================================================
# STEP 5: ADAPTIVE BAYESIAN OUT-OF-FOLD TARGET ENCODING
# ==============================================================================
ENTITY_SMOOTHING = {
    'customer_id': 5.0,        # Low shrinkage: customer return habit is persistent
    'product_id': 15.0,        # Medium shrinkage: product defect & sizing rate
    'store_id': 60.0,          # High shrinkage: macro channel policy
    'supplier_code': 40.0,
    'category_clean': 80.0,
    'promo_x_channel': 50.0,
    'promo_x_cat': 50.0
}

def apply_adaptive_oof_target_encoding(train_df, y_tr, test_df, te_configs, n_splits=5):
    """
    Computes smoothed Bayesian out-of-fold target encodings grouped by order_id.
    Zero intra-basket or lookahead leakage.
    """
    prior = float(y_tr.mean())
    names = [f'{c}_te' for c in te_configs.keys() if c in train_df.columns]
    oof_train = {n: np.full(len(train_df), prior, dtype=np.float32) for n in names}
    test_encoded = {}

    groups = train_df['order_id'].values
    gkf = GroupKFold(n_splits=n_splits)

    for col, smoothing in te_configs.items():
        if col not in train_df.columns:
            continue
        col_te_name = f'{col}_te'

        # OOF calculation on train split
        for tr_idx, val_idx in gkf.split(train_df, y_tr, groups):
            stats = pd.DataFrame({
                'key': train_df[col].values[tr_idx],
                'target': y_tr[tr_idx]
            }).groupby('key')['target'].agg(['sum', 'count'])

            smoothed = (stats['sum'] + smoothing * prior) / (stats['count'] + smoothing)
            oof_train[col_te_name][val_idx] = (
                pd.Series(train_df[col].values[val_idx]).map(smoothed).fillna(prior).values.astype(np.float32)
            )

        # Full training stats applied to test split
        full_stats = pd.DataFrame({
            'key': train_df[col].values,
            'target': y_tr
        }).groupby('key')['target'].agg(['sum', 'count'])
        full_smoothed = (full_stats['sum'] + smoothing * prior) / (full_stats['count'] + smoothing)
        test_encoded[col_te_name] = (
            pd.Series(test_df[col].values).map(full_smoothed).fillna(prior).values.astype(np.float32)
        )

    return pd.DataFrame(oof_train, index=train_df.index), pd.DataFrame(test_encoded, index=test_df.index)


# ==============================================================================
# STEP 6: PURGED ROLLING TIME-BASED CV (4 DIVERSE MODEL FAMILIES)
# ==============================================================================
print("\n--- [Step 5/7] Executing Rolling Time-Based Cross-Validation ---")

CV_FOLDS = [
    ('2024-04-01', '2024-07-01'),
    ('2024-07-01', '2024-10-01'),
    ('2024-10-01', '2025-01-02')
]

# 1. LightGBM Hyperparameters
LGB_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.035,
    'num_leaves': 63,
    'max_depth': 8,
    'min_child_samples': 250,
    'feature_fraction': 0.70,
    'bagging_fraction': 0.80,
    'bagging_freq': 1,
    'reg_alpha': 2.0,
    'reg_lambda': 12.0,
    'n_jobs': N_JOBS,
    'random_state': SEED,
    'verbose': -1
}

# 2. CatBoost Hyperparameters
CB_PARAMS = {
    'iterations': 1200,
    'learning_rate': 0.04,
    'depth': 7,
    'l2_leaf_reg': 10.0,
    'eval_metric': 'AUC',
    'random_seed': SEED,
    'early_stopping_rounds': 60,
    'verbose': 0,
    'thread_count': N_JOBS
}

fold_lgb_scores = []
fold_cb_scores = []
fold_rf_scores = []
fold_sgd_scores = []
fold_blend_scores = []
best_lgb_iterations = []

for fold_idx, (val_start, val_end) in enumerate(CV_FOLDS, 1):
    t_start = time.time()
    train_mask = (df_train['order_ts'] < val_start).values
    val_mask = ((df_train['order_ts'] >= val_start) & (df_train['order_ts'] < val_end)).values

    df_tr_fold = df_train.loc[train_mask].reset_index(drop=True)
    df_val_fold = df_train.loc[val_mask].reset_index(drop=True)

    y_tr_fold = df_tr_fold['returned'].astype(np.int32).values
    y_val_fold = df_val_fold['returned'].astype(np.int32).values

    # In-fold target encoding strictly from historical data
    te_tr_fold, te_val_fold = apply_adaptive_oof_target_encoding(
        df_tr_fold, y_tr_fold, df_val_fold, ENTITY_SMOOTHING
    )

    X_tr_fold = pd.concat([df_tr_fold[base_features], te_tr_fold], axis=1).astype(np.float32)
    X_val_fold = pd.concat([df_val_fold[base_features], te_val_fold], axis=1).astype(np.float32)

    # Model 1: LightGBM
    dtrain_fold = lgb.Dataset(X_tr_fold, label=y_tr_fold)
    dval_fold = lgb.Dataset(X_val_fold, label=y_val_fold, reference=dtrain_fold)

    lgb_fold_model = lgb.train(
        LGB_PARAMS,
        dtrain_fold,
        num_boost_round=2500,
        valid_sets=[dval_fold],
        callbacks=[lgb.early_stopping(stopping_rounds=60, verbose=False)]
    )

    p_lgb = lgb_fold_model.predict(X_val_fold, num_iteration=lgb_fold_model.best_iteration)
    score_lgb = roc_auc_score(y_val_fold, p_lgb)
    fold_lgb_scores.append(score_lgb)
    best_lgb_iterations.append(lgb_fold_model.best_iteration)

    # Model 2: CatBoost (Oblivious Trees)
    p_cb = np.zeros_like(p_lgb)
    if CATBOOST_AVAILABLE:
        cb_fold_model = CatBoostClassifier(**CB_PARAMS)
        cb_fold_model.fit(X_tr_fold, y_tr_fold, eval_set=(X_val_fold, y_val_fold), use_best_model=True)
        p_cb = cb_fold_model.predict_proba(X_val_fold)[:, 1]
        score_cb = roc_auc_score(y_val_fold, p_cb)
        fold_cb_scores.append(score_cb)
    else:
        fold_cb_scores.append(0.5)

    # Model 3: Random Forest (Bagged Variance Reducer)
    # Subsample 250k rows for fast, stable tree bagging
    rf_idx = np.random.RandomState(SEED).choice(len(X_tr_fold), size=min(250000, len(X_tr_fold)), replace=False)
    rf_fold_model = RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=50,
        max_features=0.5,
        n_jobs=N_JOBS,
        random_state=SEED
    )
    rf_fold_model.fit(X_tr_fold.iloc[rf_idx], y_tr_fold[rf_idx])
    p_rf = rf_fold_model.predict_proba(X_val_fold)[:, 1]
    score_rf = roc_auc_score(y_val_fold, p_rf)
    fold_rf_scores.append(score_rf)

    # Model 4: Gradient Descent (SGD / Logistic Regression on Scaled Features)
    scaler_fold = StandardScaler()
    X_tr_sc = np.nan_to_num(scaler_fold.fit_transform(X_tr_fold), nan=0.0)
    X_val_sc = np.nan_to_num(scaler_fold.transform(X_val_fold), nan=0.0)

    sgd_fold = SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=1e-4,
        max_iter=300,
        learning_rate='optimal',
        random_state=SEED
    )
    sgd_fold.fit(X_tr_sc, y_tr_fold)
    p_sgd = sgd_fold.predict_proba(X_val_sc)[:, 1]
    score_sgd = roc_auc_score(y_val_fold, p_sgd)
    fold_sgd_scores.append(score_sgd)

    # Percentile Rank Blend across all 4 diverse architectures
    r_lgb = rankdata(p_lgb) / len(p_lgb)
    r_cb = rankdata(p_cb) / len(p_cb) if CATBOOST_AVAILABLE else r_lgb
    r_rf = rankdata(p_rf) / len(p_rf)
    r_sgd = rankdata(p_sgd) / len(p_sgd)

    if CATBOOST_AVAILABLE:
        p_blend = 0.45 * r_lgb + 0.35 * r_cb + 0.15 * r_rf + 0.05 * r_sgd
    else:
        p_blend = 0.70 * r_lgb + 0.20 * r_rf + 0.10 * r_sgd

    score_blend = roc_auc_score(y_val_fold, p_blend)
    fold_blend_scores.append(score_blend)

    print(
        f"Fold {fold_idx} [{val_start} to {val_end}] | "
        f"LGB: {score_lgb:.4f} | CB: {score_cb:.4f} | RF: {score_rf:.4f} | "
        f"SGD: {score_sgd:.4f} | Blend: {score_blend:.4f} ({time.time() - t_start:.1f}s)"
    )

    del df_tr_fold, df_val_fold, X_tr_fold, X_val_fold, dtrain_fold, dval_fold
    gc.collect()

print("\n" + "="*55)
print("PURGED CROSS-VALIDATION SCOREBOARD (MEAN ROC-AUC)")
print("="*55)
print(f"Mean LightGBM CV       : {np.mean(fold_lgb_scores):.5f}")
if CATBOOST_AVAILABLE:
    print(f"Mean CatBoost CV       : {np.mean(fold_cb_scores):.5f}")
print(f"Mean Random Forest CV  : {np.mean(fold_rf_scores):.5f}")
print(f"Mean GradientDescent CV: {np.mean(fold_sgd_scores):.5f}")
print(f"Mean Rank Blend CV     : {np.mean(fold_blend_scores):.5f}")
print("="*55)


# ==============================================================================
# STEP 7: FULL DATASET RETRAINING & ENSEMBLE SUBMISSION
# ==============================================================================
print("\n--- [Step 6/7] Training Full Multi-Model Ensemble on All Data ---")

y_full = df_train['returned'].astype(np.int32).values

# Full target encoding
te_full, te_test = apply_adaptive_oof_target_encoding(
    df_train, y_full, df_test, ENTITY_SMOOTHING
)

X_full = pd.concat([df_train[base_features], te_full], axis=1).astype(np.float32)
X_test = pd.concat([df_test[base_features], te_test], axis=1).astype(np.float32)

# 1. Full LightGBM (Multi-seed averaging)
optimal_rounds = int(np.mean(best_lgb_iterations) * 1.15)
lgb_test_preds = np.zeros(len(df_test), dtype=np.float64)

print(f"Training LightGBM with 3 seeds ({optimal_rounds} rounds)...")
for s in [SEED, SEED + 1, SEED + 2]:
    p = dict(LGB_PARAMS, random_state=s)
    dfull_ds = lgb.Dataset(X_full, label=y_full)
    mdl_lgb = lgb.train(p, dfull_ds, num_boost_round=optimal_rounds)
    lgb_test_preds += (rankdata(mdl_lgb.predict(X_test)) / len(X_test)) / 3.0

# 2. Full CatBoost
cb_test_preds = np.zeros(len(df_test), dtype=np.float64)
if CATBOOST_AVAILABLE:
    print("Training CatBoost on full dataset...")
    full_cb = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.04,
        depth=7,
        l2_leaf_reg=10.0,
        random_seed=SEED,
        verbose=0,
        thread_count=N_JOBS
    )
    full_cb.fit(X_full, y_full)
    cb_test_preds = rankdata(full_cb.predict_proba(X_test)[:, 1]) / len(X_test)

# 3. Full Random Forest
print("Training Random Forest on full dataset...")
rf_full = RandomForestClassifier(
    n_estimators=200,
    max_depth=14,
    min_samples_leaf=40,
    max_features=0.5,
    n_jobs=N_JOBS,
    random_state=SEED
)
rf_full_idx = np.random.RandomState(SEED).choice(len(X_full), size=min(350000, len(X_full)), replace=False)
rf_full.fit(X_full.iloc[rf_full_idx], y_full[rf_full_idx])
rf_test_preds = rankdata(rf_full.predict_proba(X_test)[:, 1]) / len(X_test)

# 4. Full Gradient Descent (SGD on Scaled Matrix)
print("Training Gradient Descent (SGD) on scaled dataset...")
scaler_full = StandardScaler()
X_full_sc = np.nan_to_num(scaler_full.fit_transform(X_full), nan=0.0)
X_test_sc = np.nan_to_num(scaler_full.transform(X_test), nan=0.0)

full_sgd = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    alpha=1e-4,
    max_iter=350,
    learning_rate='optimal',
    random_state=SEED
)
full_sgd.fit(X_full_sc, y_full)
sgd_test_preds = rankdata(full_sgd.predict_proba(X_test_sc)[:, 1]) / len(X_test)

# Final Percentile Rank Ensemble Blend
if CATBOOST_AVAILABLE:
    final_predictions = 0.45 * lgb_test_preds + 0.35 * cb_test_preds + 0.15 * rf_test_preds + 0.05 * sgd_test_preds
else:
    final_predictions = 0.70 * lgb_test_preds + 0.20 * rf_test_preds + 0.10 * sgd_test_preds


# ==============================================================================
# SUBMISSION GENERATION & VERIFICATION
# ==============================================================================
print("\n--- [Step 7/7] Generating & Validating submission.csv ---")

submission = pd.DataFrame({
    'order_line_id': df_test['order_line_id'],
    'returned': final_predictions
})

submission.to_csv('submission4.csv', index=False)
print("Saved submission4.csv successfully!")
print(f"Total Rows       : {len(submission):,}")
print(f"Missing Values   : {submission['returned'].isnull().sum()}")
print(f"Value Range      : [{submission['returned'].min():.5f}, {submission['returned'].max():.5f}]")
print("\nFirst 10 Predictions:")
print(submission.head(10))